In [115]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from data_loader import load_weekly_data, load_supplementary
df = load_weekly_data()

In [116]:
df1 = load_supplementary()

df1['pass_completed'] = df1['pass_result'].apply(lambda x: 1 if x=='C' else 0)

df1 = df1[["game_id", "play_id", "down", "yards_to_go", "yardline_number", "defenders_in_the_box", "dropback_distance", "pass_completed"]]

df1.head(3)

,game_id,play_id,down,yards_to_go,yardline_number,defenders_in_the_box,dropback_distance,pass_completed
0,2023090700,3461,3,12,23,6,5.30,1
1,2023090700,461,1,10,34,7,4.72,1
2,2023090700,1940,2,10,42,6,4.44,0


In [117]:
df2 = load_weekly_data()

df2 = df2[["game_id", "play_id", "player_name", "player_role", "x", "y", "s", "a", "o"]]

df_qb_wr = df2[df2['player_role'].isin(['Passer', 'Targeted Receiver'])]

qb_wr_wide = df_qb_wr.pivot_table(
        index=["game_id", "play_id"],
        columns="player_role",
        values=["x", "y", "o", "s", "a"],
        aggfunc="last"
    )

qb_wr_wide.columns = [f"{v}_{r}" for v, r in qb_wr_wide.columns]
qb_wr_wide = qb_wr_wide.reset_index()

qb_wr_wide["distance_qb_wr"] = np.sqrt(
        (qb_wr_wide["x_Passer"] - qb_wr_wide["x_Targeted Receiver"])**2 +
        (qb_wr_wide["y_Passer"] - qb_wr_wide["y_Targeted Receiver"])**2
    )

qb_wr_wide["orientation_diff"] = np.abs(qb_wr_wide["o_Passer"] - qb_wr_wide["o_Targeted Receiver"])

qb_wr_wide = qb_wr_wide.rename(columns={
        "s_Targeted Receiver": "wr_speed",
        "a_Targeted Receiver": "wr_accel"
    })

qb_wr_wide = qb_wr_wide[["game_id", "play_id", "distance_qb_wr", "orientation_diff", "wr_speed", "wr_accel"]]
df_qb_wr = df_qb_wr[["game_id", "play_id", "player_name", "player_role"]] 

model_df = df_qb_wr.merge(
        qb_wr_wide,
        on=["game_id", "play_id"],
        how="inner"
    )

In [118]:
model_df = model_df.drop_duplicates(subset="play_id", keep="first")

model_df = model_df.merge(
        df1,
        on=["game_id", "play_id"],
        how="inner"
    )

model_df.head(3)

,game_id,play_id,player_name,player_role,distance_qb_wr,orientation_diff,wr_speed,wr_accel,down,yards_to_go,yardline_number,defenders_in_the_box,dropback_distance,pass_completed
0,2023090700,101,Jared Goff,Passer,23.257319,105.45,7.90,2.68,3,3,32,6,2.13,0
1,2023090700,194,Patrick Mahomes,Passer,11.401895,28.93,6.09,2.14,3,2,21,6,3.86,1
2,2023090700,219,Patrick Mahomes,Passer,16.136127,98.16,3.85,2.77,1,10,31,6,2.37,1


In [119]:
model_df = model_df.dropna()

In [128]:
model_df.to_csv('knn.csv', index=False) 

In [120]:
y = model_df["pass_completed"]

X = model_df[[
    "distance_qb_wr",
    "orientation_diff",
    "dropback_distance",
    "wr_speed",
    "wr_accel"
]]

In [121]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [122]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(weights="distance"))
])

In [123]:
param_grid = {"knn__n_neighbors": range(1, 41, 2)}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="balanced_accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

,estimator,Pipeline(step...'distance'))])
,param_grid,"{'knn__n_neighbors': range(1, 41, 2)}"
,scoring,'balanced_accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [124]:
results_df = pd.DataFrame(grid.cv_results_)

results_df["k"] = results_df["param_knn__n_neighbors"]
results_df["mean_score"] = results_df["mean_test_score"]

best_k = grid.best_params_["knn__n_neighbors"]
best_score = grid.best_score_

In [125]:
pipe2 = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k,
    weights="distance"))
])

In [126]:
pipe2.fit(X_train, y_train)
y_pred = pipe2.predict(X_test)

In [127]:
acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Accuracy: {acc:.3f}")
print(f"Balanced accuracy: {bal_acc:.3f}")

Accuracy: 0.705
Balanced accuracy: 0.578
